# AutoResearch FetchSlide — Colab T4 GPU

Run the from-scratch FetchSlide RL loop on a Colab T4 GPU. Same code as the local `autoresearch` package; only the device changes (`cuda`).

1. Run all cells in order.
2. Cell 2 clones the repo.
3. Cell 3 runs the reference-recipe training (pure sparse + TD3 + reference cadence) on T4.
4. Cell 4 runs the multi-agent autoresearch loop.

All checkpoints and summaries are written under `AUTORESEARCH_RUNS` (default `/content/autoresearch-runs`).

In [ ]:
!pip -q install gymnasium gymnasium-robotics torch tensorboard imageio imageio-ffmpeg 2>&1 | tail -3

In [2]:
import torch
assert torch.cuda.is_available(), 'CUDA GPU required: Runtime > Change runtime type > T4 GPU'
print('CUDA device:', torch.cuda.get_device_name(0))
assert 'T4' in torch.cuda.get_device_name(0) or torch.cuda.get_device_capability(0)[0] >= 7, 'Expected a T4-class CUDA GPU'


CUDA device: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess
REPO = '/content/autoresearch'
RUNS = '/content/drive/MyDrive/autoresearch-runs'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/DeconvFFT/fetch-and-slide-HRE-PRE.git', REPO], check=True)
os.makedirs(RUNS, exist_ok=True)
os.environ['AUTORESEARCH_RUNS'] = RUNS
sys.path.insert(0, REPO)
print('REPO', REPO, 'RUNS', RUNS)


## Reference-recipe training (pure sparse + exact DDPG + reference cadence)

This matches the reference HER+DDPG recipe that hit 80%: pure sparse reward, TD3 (better than DDPG per literature), and the reference cadence (rollouts_per_cycle=2, optimsteps=40). Runs on T4 GPU.

In [ ]:
# Exact DDPG reference configuration. Training is run in resumable chunks below.
TOTAL_EPISODES = 20000
CHUNK_EPISODES = 100
BASE_CONFIG = {
    'algorithm': 'ddpg', 'horizon': 50, 'eval_episodes': 50, 'batch_size': 256,
    'warmup_steps': 0, 'updates_per_step': 1, 'eval_every': 2000, 'log_every': 100,
    'device': 'cuda', 'her_future': 4, 'her_ratio': 0.8, 'per': False, 'hper': False,
    'dense_reward': False, 'success_bonus': 0.0, 'reach_coef': 0.0,
    'reach_contact_bonus': 0.0, 'push_coef': 0.0, 'goal_bonus': 0.0,
    'goal_bonus_radius': 0.4, 'actor_l2': 1.0, 'scripted_rollouts': 0,
    'scripted_every': 0, 'rollouts_per_cycle': 2, 'optimsteps': 40,
    'replay_capacity': 1000000,
}
print('exact DDPG total steps:', TOTAL_EPISODES * BASE_CONFIG['horizon'])
print('chunk size:', CHUNK_EPISODES * BASE_CONFIG['horizon'], 'steps')


In [ ]:
from pathlib import Path
import os, subprocess, sys
controller_path = Path(RUNS) / 'ref_recipe_controller.py'
os.environ['AUTORESEARCH_TOTAL_EPISODES'] = '20000'
os.environ['AUTORESEARCH_CHUNK_EPISODES'] = '100'
controller_path.write_text('import json, os, subprocess, sys, fcntl\nfrom pathlib import Path\n\nREPO = os.environ[\'REPO\'] if \'REPO\' in os.environ else \'/content/autoresearch\'\nRUNS = Path(os.environ[\'AUTORESEARCH_RUNS\'])\nTOTAL_EPISODES = int(os.environ.get(\'AUTORESEARCH_TOTAL_EPISODES\', \'20000\'))\nCHUNK_EPISODES = int(os.environ.get(\'AUTORESEARCH_CHUNK_EPISODES\', \'100\'))\nROOT = RUNS / \'ref_recipe_chunks\'\nROOT.mkdir(parents=True, exist_ok=True)\nBASE = {\n    \'algorithm\': \'ddpg\', \'horizon\': 50, \'eval_episodes\': 50, \'batch_size\': 256,\n    \'warmup_steps\': 0, \'updates_per_step\': 1, \'eval_every\': 2000, \'log_every\': 100,\n    \'device\': \'cuda\', \'her_future\': 4, \'her_ratio\': 0.8, \'per\': False, \'hper\': False,\n    \'dense_reward\': False, \'success_bonus\': 0.0, \'reach_coef\': 0.0,\n    \'reach_contact_bonus\': 0.0, \'push_coef\': 0.0, \'goal_bonus\': 0.0,\n    \'goal_bonus_radius\': 0.4, \'actor_l2\': 1.0, \'scripted_rollouts\': 0,\n    \'scripted_every\': 0, \'rollouts_per_cycle\': 2, \'optimsteps\': 40,\n    \'replay_capacity\': 1000000,\n}\nLOG = RUNS / \'ref_recipe_controller.log\'\nPID = RUNS / \'ref_recipe_controller.pid\'\nLOCK = RUNS / "ref_recipe_controller.lock"\nlock_file = LOCK.open("w")\ntry:\n    fcntl.flock(lock_file.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)\nexcept BlockingIOError:\n    print("controller lock already held; exiting duplicate controller")\n    raise SystemExit(0)\n\ndef completed_chunks():\n    return sorted(d for d in ROOT.glob(\'chunk-*\') if (d / \'metrics.json\').exists() and (d / \'checkpoint.pt\').exists())\n\ndone = completed_chunks()\nepisodes_done = sum(json.loads((d / \'config.json\').read_text())[\'train_episodes\'] for d in done)\nprevious = done[-1] / \'checkpoint.pt\' if done else None\nwith LOG.open(\'a\', buffering=1) as controller_log:\n    controller_log.write(f\'controller start: episodes_done={episodes_done}/{TOTAL_EPISODES}\\n\')\n    chunk_index = len(done)\n    while episodes_done < TOTAL_EPISODES:\n        episodes = min(CHUNK_EPISODES, TOTAL_EPISODES - episodes_done)\n        out = ROOT / f\'chunk-{chunk_index:05d}\'\n        cfg = dict(BASE)\n        cfg[\'train_episodes\'] = episodes\n        cfg[\'seed\'] = 7 + episodes_done\n        cfg_path = out / \'config.json\'\n        out.mkdir(parents=True, exist_ok=True)\n        cfg_path.write_text(json.dumps(cfg, indent=2))\n        cmd = [sys.executable, \'-m\', \'autoresearch.worker\', \'--config\', str(cfg_path), \'--output\', str(out)]\n        if previous is not None:\n            cmd += [\'--init-checkpoint\', str(previous)]\n        controller_log.write(f\'chunk {chunk_index}: episodes={episodes}, steps={episodes*50}, init={previous}\\n\')\n        env = dict(os.environ)\n        env[\'PYTHONPATH\'] = REPO + os.pathsep + env.get(\'PYTHONPATH\', \'\')\n        proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\n        for line in proc.stdout:\n            controller_log.write(f\'[chunk {chunk_index}] {line}\')\n        rc = proc.wait()\n        controller_log.write(f\'chunk {chunk_index} exit={rc}\\n\')\n        if rc != 0 or not (out / \'metrics.json\').exists() or not (out / \'checkpoint.pt\').exists():\n            controller_log.write(\'controller stopped; rerun launch cell to retry this chunk\\n\')\n            break\n        if chunk_index % int(os.environ.get("AUTORESEARCH_RENDER_EVERY", "1")) == 0:\n            render_out = out / "render"\n            render_cmd = [\n                sys.executable, "-m", "autoresearch.render_checkpoint",\n                "--config", str(cfg_path), "--checkpoint", str(out / "checkpoint.pt"),\n                "--output", str(render_out), "--tensorboard", str(RUNS / "tensorboard"),\n                "--episodes", "2", "--step", str((episodes_done + episodes) * 50),\n            ]\n            controller_log.write(f"render chunk {chunk_index}: start\\n")\n            try:\n                render_proc = subprocess.run(render_cmd, cwd=REPO, env=env, stdout=controller_log, stderr=subprocess.STDOUT, text=True, timeout=180)\n                controller_log.write(f"render chunk {chunk_index}: exit={render_proc.returncode}\\n")\n            except subprocess.TimeoutExpired:\n                controller_log.write(f"render chunk {chunk_index}: timeout; continuing training\\n")\n        episodes_done += episodes\n        previous = out / \'checkpoint.pt\'\n        chunk_index += 1\n    controller_log.write(f\'controller finished: episodes_done={episodes_done}/{TOTAL_EPISODES}\\n\')\n')
controller_log = Path(RUNS) / 'ref_recipe_controller.log'
controller_pid = Path(RUNS) / 'ref_recipe_controller.pid'
already_running = False
if controller_pid.exists():
    try:
        old_pid = int(controller_pid.read_text())
        os.kill(old_pid, 0)
        already_running = True
        print('controller already running; pid=', old_pid)
    except (ProcessLookupError, ValueError, PermissionError):
        pass
if not already_running:
    log = open(controller_log, 'a', buffering=1)
    proc = subprocess.Popen([sys.executable, str(controller_path)], cwd=REPO, env=dict(os.environ, PYTHONPATH=REPO), stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    log.close()
    controller_pid.write_text(str(proc.pid))
    print('resumable controller started; pid=', proc.pid)
    print('controller log:', controller_log)
    print('rerun the monitor cell after reconnecting')


In [ ]:
import time, os, json, re
from pathlib import Path
from IPython.display import clear_output
log_path = Path(RUNS) / 'ref_recipe_controller.log'
pid_path = Path(RUNS) / 'ref_recipe_controller.pid'
TOTAL_CHUNKS = 200
while True:
    clear_output(wait=True)
    print('=== FetchSlide exact DDPG controller ===')
    text = log_path.read_text() if log_path.exists() else ''
    lines = text.splitlines()
    if lines:
        print('\n'.join(lines[-30:]))
        print(f'log age: {time.time() - log_path.stat().st_mtime:.0f}s')
    else:
        print('controller log not created yet')
    chunks = sorted((Path(RUNS) / 'ref_recipe_chunks').glob('chunk-*'))
    complete = [c for c in chunks if (c / 'metrics.json').exists() and (c / 'checkpoint.pt').exists()]
    active = [c for c in chunks if c not in complete]
    print(f'completed chunks: {len(complete)} / {TOTAL_CHUNKS}')
    print('latest completed:', complete[-1].name if complete else 'none')
    print('active chunk:', active[-1].name if active else 'none')
    if pid_path.exists():
        pid = int(pid_path.read_text())
        try:
            os.kill(pid, 0)
            print('controller status: RUNNING, pid=', pid)
        except (ProcessLookupError, PermissionError):
            print('controller status: EXITED')
    if not pid_path.exists():
        break
    time.sleep(30)


## Live physics dashboard

TensorBoard reads persistent Drive logs while the controller runs. Rendered checkpoint videos are under each chunk `render/` directory.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/autoresearch-runs/tensorboard

## Multi-agent autoresearch loop

Runs the Karpathy-style autonomous loop with the two-model proposal system (pro strategist + flash implementor) on T4. Requires an OpenRouter API key.

In [ ]:
import os
os.environ['OPENROUTER_API_KEY'] = 'YOUR_OPENROUTER_KEY_HERE'  # <-- set your key
print('key set:', bool(os.environ['OPENROUTER_API_KEY'] and os.environ['OPENROUTER_API_KEY'] != 'YOUR_OPENROUTER_KEY_HERE'))

In [ ]:
import subprocess, sys, os
# Run multi-agent improvements from the exact DDPG baseline on CUDA.
os.environ['AUTORESEARCH_ALGORITHM'] = 'ddpg'
os.environ['AUTORESEARCH_TRAIN_EPISODES'] = '2000'
os.environ['AUTORESEARCH_MAX_EPISODES'] = '2000'
os.environ['AUTORESEARCH_EVAL_EPISODES'] = '10'
os.environ['AUTORESEARCH_BATCH_SIZE'] = '256'
os.environ['AUTORESEARCH_DEVICE'] = 'cuda'
os.environ['AUTORESEARCH_DENSE_REWARD'] = 'false'
os.environ['AUTORESEARCH_HER_FUTURE'] = '4'
os.environ['AUTORESEARCH_ROLLOUTS_PER_CYCLE'] = '2'
os.environ['AUTORESEARCH_OPTIMSTEPS'] = '40'
env = dict(os.environ); env['PYTHONPATH'] = REPO + os.pathsep + env.get('PYTHONPATH', '')
agent_log = f'{RUNS}/multi_agent.log'
agent_pid = f'{RUNS}/multi_agent.pid'
log = open(agent_log, 'a', buffering=1)
proc = subprocess.Popen([sys.executable, '-m', 'autoresearch.agent_loop', '--tag', 'colab-ddpg', '--iterations', '5'],
                       cwd=REPO, env=env, stdout=log, stderr=subprocess.STDOUT,
                       start_new_session=True)
log.close()
open(agent_pid, 'w').write(str(proc.pid))
print('multi-agent loop started in background; pid=', proc.pid)
print('log:', agent_log)


In [ ]:
from pathlib import Path
import os, json
log_path = Path(RUNS) / 'ref_recipe_controller.log'
pid_path = Path(RUNS) / 'ref_recipe_controller.pid'
if log_path.exists():
    print(log_path.read_text()[-10000:])
else:
    print('controller log not created yet')
if pid_path.exists():
    pid = int(pid_path.read_text())
    try:
        os.kill(pid, 0)
        print('controller status: RUNNING, pid=', pid)
    except ProcessLookupError:
        print('controller status: EXITED')
chunks = sorted((Path(RUNS) / 'ref_recipe_chunks').glob('chunk-*'))
print('completed chunks:', sum((c / 'metrics.json').exists() for c in chunks), '/', len(chunks))


In [ ]:
from pathlib import Path
import os, glob
log_path = Path(RUNS) / 'multi_agent.log'
pid_path = Path(RUNS) / 'multi_agent.pid'
if log_path.exists():
    print(log_path.read_text()[-8000:])
else:
    print('multi-agent log not created yet')
if pid_path.exists():
    pid = int(pid_path.read_text())
    try:
        os.kill(pid, 0)
        print('multi-agent status: RUNNING, pid=', pid)
    except ProcessLookupError:
        print('multi-agent status: EXITED')
best = sorted(glob.glob(f'{RUNS}/run-*/best_checkpoint.pt'))[-1] if glob.glob(f'{RUNS}/run-*/best_checkpoint.pt') else None
print('best checkpoint:', best)
print('results.tsv:')
!cd {REPO} && cat results.tsv 2>/dev/null | tail -20
